In [ ]:
from scapy.all import *
import time

arp_tabla = {}
ultimo_visto = {}

def alert(pkt):
    if pkt.haslayer(ARP) and pkt[ARP].op == 2:
        ip = pkt[ARP].psrc
        mac = pkt[ARP].hwsrc

        now = time.time()

        # detecta cambios de la MAC
        if ip in arp_tabla and arp_tabla[ip] != mac:
            print("\nARP SPOOFING DETECTADO !!!")
            print(f"IP: {ip}")
            print(f"MAC original: {arp_tabla[ip]}")
            print(f"MAC nueva: {mac}")
            print("----------------------------")

        # detecta muchas repeticiones sospechosas en poco tiempo
        if ip in ultimo_visto and now - ultimo_visto[ip] < 1:
            print(f"Posible ARP Spoofing detectado!! (ARP reply anómola): {ip}")

        arp_tabla[ip] = mac
        ultimo_visto[ip] = now

print("[*] Monitorizando ARP...")
sniff(iface="eth0", filter="arp", prn=alert, store=0)